# 06_1d_empymod_results

Visualize and export results from 1D empymod ensemble inversions (`OneDInversionRunN`).

Panels:
- Pseudo-2D mean and std sections (with true model overlay).
- Per-Tx layered model diagnostics (mean +/- std).
- Modelled empymod data mean +/- std vs observed.
- Export selected run to SEGY / NPZ.

In [ ]:
from pathlib import Path
import json
import os
import re
import signal
import sys
import traceback

import numpy as np
from IPython.display import display, clear_output

try:
    import ipywidgets as ipw
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
except Exception as exc:
    raise RuntimeError(
        "Missing dependencies. Install with: pip install ipywidgets plotly numpy"
    ) from exc

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.modules.segy import write_resistivity_to_segy_from_template
from scripts.modules.fd_visualization import compute_amp_phase_for_fd_outputs
from scripts.modules.empymod_1d_forward import forward_empymod_1d_layered_amp_phase, wrap_phase_rad
from third_party.rockseis.io.rsfile import rsfile

FDMODEL_DIR = ROOT / "FDmodel"
INV_INPUT_DIR = ROOT / "InversionInput"
ONED_BASE_DIR = ROOT / "Results" / "Empymod1D"
LEGACY_ONED_BASE_DIR = FDMODEL_DIR / "OneDInversion"
SG_TRUE_PATH = FDMODEL_DIR / "sg.rss"
SETUP_META = FDMODEL_DIR / "setup_metadata.json"
ONED_RUN_DIR_PATTERN = re.compile(r"^OneDInversionRun(\d+)$")
VOILA_PID_FILE = ROOT / ".voila_1d_results_server.pid"

state = {
    "messages": [],
    "run_data": None,
    "summary": None,
    "real_result": None,
    "syn_result": None,
}


def push_message(msg):
    state["messages"].append(str(msg))
    if len(state["messages"]) > 15:
        state["messages"] = state["messages"][-15:]
    status_out.value = "\n".join(state["messages"])


def list_1d_run_dirs():
    if not ONED_BASE_DIR.exists():
        return []
    out = []
    for child in ONED_BASE_DIR.iterdir():
        if child.is_dir():
            m = ONED_RUN_DIR_PATTERN.match(child.name)
            if m:
                out.append((int(m.group(1)), child))
    out.sort(key=lambda x: x[0])
    return out


def _read_rss_model(path):
    f = rsfile()
    f.read(str(path))
    data = np.asarray(f.data, dtype=float)
    data = np.squeeze(data)
    if data.ndim != 2:
        raise ValueError(f"Expected 2D model, got shape {data.shape}")
    nx, nz = int(data.shape[0]), int(data.shape[1])
    grid = np.asarray(data.T, dtype=float)
    dx = float(f.geomD[0]) if f.geomD[0] else 1.0
    ox = float(f.geomO[0])
    iz = 2 if (len(f.geomN) > 2 and int(f.geomN[2]) > 0) else 1
    dz = float(f.geomD[iz]) if f.geomD[iz] else 1.0
    oz = float(f.geomO[iz])
    x = ox + dx * np.arange(nx)
    z = oz + dz * np.arange(nz)
    return x, z, grid


def conductivity_to_resistivity(grid, min_sigma=1e-12):
    sigma = np.clip(np.asarray(grid, dtype=float), min_sigma, 1e12)
    return 1.0 / sigma


def parse_freqs(text):
    parts = [p.strip() for p in str(text).split(',') if p.strip()]
    if not parts:
        raise ValueError('Frequency list cannot be empty.')
    vals = np.asarray([float(p) for p in parts], dtype=float)
    if np.any(vals <= 0):
        raise ValueError('Frequencies must be positive.')
    return vals


def get_real_data_paths():
    # Try preferred outputs first, then InversionInput fallback.
    candidates = [
        (FDMODEL_DIR / 'Data' / 'Hxshot.rss', FDMODEL_DIR / 'Data' / 'Hzshot.rss'),
        (FDMODEL_DIR / 'Data' / 'Hx_data.rss', FDMODEL_DIR / 'Data' / 'Hz_data.rss'),
        (INV_INPUT_DIR / 'Hx_data.rss', INV_INPUT_DIR / 'Hz_data.rss'),
    ]
    for hx, hz in candidates:
        if hx.exists() and hz.exists():
            return hx, hz
    # Return first pair so errors upstream are explicit.
    return candidates[0]


def _resolve_segy_template_path():
    if SETUP_META.exists():
        try:
            meta = json.loads(SETUP_META.read_text())
            p_raw = meta.get("segy_template_path")
            if p_raw:
                p = Path(p_raw).expanduser()
                if not p.is_absolute():
                    p = (ROOT / p).resolve()
                if p.exists():
                    return p, "setup_metadata.json"
        except Exception:
            pass
    fallback = ROOT / "input.segy"
    if fallback.exists():
        return fallback, "fallback ROOT/input.segy"
    return None, None


def load_run_data(run_dir):
    run_dir = Path(run_dir)
    npz_path = run_dir / "empymod_1d_phase_inversion_results.npz"
    json_path = run_dir / "empymod_1d_phase_inversion_summary.json"
    legacy_mode = False
    if not npz_path.exists():
        # Backward compatibility: older layout stored artifacts directly under
        # FDmodel/OneDInversion/ instead of FDmodel/OneDInversion/OneDInversionRunN/.
        legacy_npz = LEGACY_ONED_BASE_DIR / "empymod_1d_phase_inversion_results.npz"
        legacy_json = LEGACY_ONED_BASE_DIR / "empymod_1d_phase_inversion_summary.json"
        if legacy_npz.exists():
            npz_path = legacy_npz
            json_path = legacy_json
            legacy_mode = True
        else:
            raise FileNotFoundError(f"NPZ not found: {npz_path}")
    data = dict(np.load(npz_path, allow_pickle=True))
    summary = {}
    if json_path.exists():
        summary = json.loads(json_path.read_text())
    if legacy_mode:
        summary["legacy"] = True
        summary["legacy_base"] = str(LEGACY_ONED_BASE_DIR)
    return data, summary


def _rebuild_pseudosection_from_saved_layers(run_data, summary, background_rho_override=None):
    """Rebuild pseudo-2D mean/std sections from saved per-Tx layer ensembles."""
    tx_x = np.asarray(run_data.get('tx_x', []), dtype=float)
    tx_z = np.asarray(run_data.get('tx_z', []), dtype=float)
    rho_layers = np.asarray(run_data.get('rho_layers', []), dtype=float)
    thk_layers = np.asarray(run_data.get('thickness_layers', []), dtype=float)
    if tx_x.size == 0 or rho_layers.size == 0 or thk_layers.size == 0:
        raise ValueError('Missing saved layer arrays to rebuild pseudo-section.')

    rho_layers_std = run_data.get('rho_layers_std', None)
    rho_layers_std = np.asarray(rho_layers_std, dtype=float) if rho_layers_std is not None else None

    if background_rho_override is None:
        background_rho = float((summary or {}).get('background_rho', 10.0))
    else:
        background_rho = float(background_rho_override)

    # Axes from RSS templates.
    axes_candidates = [
        (FDMODEL_DIR / 'sg0.rss'),
        (FDMODEL_DIR / 'sg_ls.rss'),
        (SG_TRUE_PATH),
    ]
    x_grid = z_grid = None
    for p in axes_candidates:
        if p.exists():
            x_grid, z_grid, _g = _read_rss_model(p)
            break
    if x_grid is None or z_grid is None:
        raise FileNotFoundError('No RSS axis template found (sg0.rss/sg_ls.rss/sg.rss).')
    x_grid = np.asarray(x_grid, dtype=float)
    z_grid = np.asarray(z_grid, dtype=float)

    # Infer halfspan from thickness sums.
    spans = []
    for i in range(thk_layers.shape[0]):
        v = np.asarray(thk_layers[i, :], dtype=float)
        v = v[np.isfinite(v)]
        if v.size:
            spans.append(float(np.sum(v)))
    if not spans:
        raise ValueError('Could not infer depth window from thickness layers.')
    halfspan = 0.5 * float(np.median(np.asarray(spans, dtype=float)))
    z_start_rel = -halfspan
    z_end_rel = halfspan

    # Sort tx by x for interpolation.
    order = np.argsort(tx_x)
    tx_x_s = tx_x[order]
    tx_z_s = tx_z[order] if tx_z.size else tx_z
    rho_s = rho_layers[order, :]
    thk_s = thk_layers[order, :]
    rho_std_s = rho_layers_std[order, :] if rho_layers_std is not None else None

    n_tx = int(rho_s.shape[0])
    mean_profiles = np.full((n_tx, z_grid.size), background_rho, dtype=float)
    std_profiles = np.zeros((n_tx, z_grid.size), dtype=float)

    # Build per-tx layer->z profiles (std uses rho std only).
    for i in range(n_tx):
        rho_row = np.asarray(rho_s[i, :], dtype=float)
        thk_row = np.asarray(thk_s[i, :], dtype=float)
        rho_std_row = np.asarray(rho_std_s[i, :], dtype=float) if rho_std_s is not None else None

        rho_mask = np.isfinite(rho_row)
        rho_valid = rho_row[rho_mask]
        if rho_valid.size == 0:
            continue
        n_layers = int(rho_valid.size)

        thk_valid = thk_row[np.isfinite(thk_row)]
        if thk_valid.size >= (n_layers - 1):
            thk_valid = thk_valid[: n_layers - 1]
        else:
            n_layers = int(thk_valid.size + 1)
            rho_valid = rho_valid[:n_layers]

        z_top_abs = float(tx_z_s[i]) + z_start_rel
        z_bottom_abs = float(tx_z_s[i]) + z_end_rel
        interfaces_abs = float(tx_z_s[i]) + (z_start_rel + np.cumsum(thk_valid, dtype=float)) if thk_valid.size else np.array([], dtype=float)

        mean_prof = np.full_like(z_grid, background_rho, dtype=float)
        std_prof = np.zeros_like(z_grid, dtype=float)

        rho_std_valid = None
        if rho_std_row is not None:
            rho_std_valid = rho_std_row[rho_mask][:n_layers]

        z0 = z_top_abs
        for li in range(n_layers):
            if li < interfaces_abs.size:
                z1 = float(interfaces_abs[li])
                m = (z_grid >= z0) & (z_grid < z1)
            else:
                z1 = z_bottom_abs
                m = (z_grid >= z0) & (z_grid <= z1)
            mean_prof[m] = float(rho_valid[li])
            if rho_std_valid is not None:
                sv = rho_std_valid[li]
                std_prof[m] = float(sv) if np.isfinite(sv) else 0.0
            z0 = z1

        mean_profiles[i, :] = mean_prof
        std_profiles[i, :] = std_prof

    # Interpolate along x for each z.
    section_rho = np.zeros((z_grid.size, x_grid.size), dtype=float)
    section_rho_std = np.zeros((z_grid.size, x_grid.size), dtype=float)
    for iz in range(z_grid.size):
        section_rho[iz, :] = np.interp(x_grid, tx_x_s, mean_profiles[:, iz], left=background_rho, right=background_rho)
        # Simplified std interpolation: interpolate std itself and assume 0 outside tx span.
        section_rho_std[iz, :] = np.interp(x_grid, tx_x_s, std_profiles[:, iz], left=0.0, right=0.0)

    return x_grid, z_grid, section_rho, section_rho_std


# rebuilt pseudo section from saved layers (helper)
# -------------------- UI --------------------
runs = list_1d_run_dirs()
run_options = [(p.name, str(p)) for _, p in runs] if runs else [("No runs", "")]

run_selector = ipw.Dropdown(
    options=run_options,
    value=run_options[-1][1] if run_options and run_options[-1][1] else "",
    description="Run:",
    layout=ipw.Layout(width="420px"),
)
load_btn = ipw.Button(description="Load run", button_style="primary")
refresh_runs_btn = ipw.Button(description="Refresh run list")

section_out = ipw.Output(layout=ipw.Layout(width="100%", height="980px"))
x_slice_input = ipw.FloatText(value=0.0, description="x slice (m):", layout=ipw.Layout(width="260px"))
z_slice_input = ipw.FloatText(value=0.0, description="z slice (m):", layout=ipw.Layout(width="260px"))
bg_rho_input = ipw.FloatText(value=10.0, description="bg rho (Ohm.m):", layout=ipw.Layout(width="260px"))
rebuild_section_btn = ipw.Button(description="Rebuild section from saved layers", button_style="warning")
rho_z_slice_out = ipw.Output(layout=ipw.Layout(width="100%", height="340px"))
rho_x_slice_out = ipw.Output(layout=ipw.Layout(width="100%", height="340px"))
tx_diag_out = ipw.Output(layout=ipw.Layout(width="100%", height="520px"))
data_compare_out = ipw.Output(layout=ipw.Layout(width="100%", height="520px"))

tx_select = ipw.Dropdown(options=[("n/a", -1)], value=-1, description="Tx")
refresh_tx_btn = ipw.Button(description="Refresh Tx plots")
uncertainty_select = ipw.Dropdown(options=[("1 std", "std"), ("90% CI", "ci90")], value="std", description="Uncertainty")

freqs_input = ipw.Text(value=",".join([f"{v:g}" for v in [2000.0, 4000.0, 6000.0]]), description="Freqs (Hz):", layout=ipw.Layout(width="320px"))
load_real_btn = ipw.Button(description="Load real data", button_style="primary")
generate_syn_btn = ipw.Button(description="Generate synthetics from inverted model", button_style="success")

# Standard data-compare controls (mirrors 04_results.ipynb)
component_select = ipw.Dropdown(options=[("Hx", "Hx"), ("Hz", "Hz")], value="Hx", description="component")
metric_select = ipw.Dropdown(
    options=[
        ("Amplitude vs rx (local index)", "amp_vs_rx"),
        ("Phase vs rx (deg, local index)", "phase_vs_rx_deg"),
        ("Amplitude vs tx (fixed local rx)", "amp_vs_tx"),
        ("Phase vs tx (deg, fixed local rx)", "phase_vs_tx_deg"),
        ("Amplitude vs frequency", "amp_vs_freq"),
        ("Phase vs frequency (deg)", "phase_vs_freq_deg"),
    ],
    value="amp_vs_rx",
    description="plot",
)
comp_freq = ipw.Dropdown(options=[("n/a", 0)], value=0, description="frequency")
rx_local_select = ipw.IntSlider(value=0, min=0, max=0, step=1, description="local rx", continuous_update=False, layout=ipw.Layout(width="260px"))
trace_idx = ipw.IntSlider(value=0, min=0, max=0, step=1, description="trace idx", continuous_update=False, layout=ipw.Layout(width="320px"))

export_segy_btn = ipw.Button(description="Export mean section to SEGY")
export_status = ipw.HTML(value="")
quit_btn = ipw.Button(description="Quit GUI server", button_style="danger", layout=ipw.Layout(width="170px"))
status_out = ipw.Textarea(value="Ready.", description="Status", layout=ipw.Layout(width="100%", height="160px"))


def refresh_run_options():
    runs = list_1d_run_dirs()
    opts = [(p.name, str(p)) for _, p in runs] if runs else [("No runs", "")]
    run_selector.options = opts
    if opts and opts[-1][1]:
        run_selector.value = opts[-1][1]


def on_load_run(_):
    try:
        run_dir = run_selector.value
        if not run_dir:
            push_message("No run selected.")
            return
        data, summary = load_run_data(run_dir)
        state["run_data"] = data
        state["summary"] = summary
        state["run_dir"] = run_dir
        state["force_rebuild_section"] = False
        try:
            bg_default = float(summary.get("background_rho", 10.0))
            if np.isfinite(bg_default):
                bg_rho_input.value = float(bg_default)
        except Exception:
            bg_rho_input.value = 10.0

        tx_ids = data.get("tx_ids", np.array([])).astype(int)
        if tx_ids.size:
            tx_select.options = [(f"Tx {int(t)}", int(t)) for t in tx_ids]
            tx_select.value = int(tx_ids[0])
        else:
            tx_select.options = [("n/a", -1)]
            tx_select.value = -1

        n_runs = int(data.get("n_runs", 1))
        push_message(f"Loaded {Path(run_dir).name}: {tx_ids.size} Tx, {n_runs} run(s)/Tx.")

        # Standard data-compare UI is populated after "Load real data"
        # (real_result provides both geometry and the frequency axis).

        update_section_plot()
        update_tx_diag()
        update_data_compare()
    except Exception as exc:
        push_message(f"Load failed: {exc}")
        push_message(traceback.format_exc())


def on_rebuild_section(_):
    if state.get('run_data') is None:
        push_message('Load a run first.')
        return
    state['force_rebuild_section'] = True
    update_section_plot()


def update_section_plot():
    data = state.get("run_data")
    if data is None:
        return
    with section_out:
        clear_output(wait=True)
        try:
            sec_x = data.get("section_x")
            sec_z = data.get("section_z")
            sec_rho = data.get("section_rho")
            sec_std = data.get("section_rho_std")

            if sec_x is None or sec_z is None or sec_rho is None:
                display(ipw.HTML("No pseudo-2D section in this run. Build section in 05 first."))
                return

            has_std = sec_std is not None and np.any(np.isfinite(np.asarray(sec_std, dtype=float)))

            rho_true = None
            x_true = z_true = None
            if SG_TRUE_PATH.exists():
                try:
                    x_true, z_true, g_true = _read_rss_model(SG_TRUE_PATH)
                    rho_true = conductivity_to_resistivity(g_true)
                except Exception:
                    pass

            ncols = 1 + (1 if has_std else 0) + (1 if rho_true is not None else 0)
            titles = ["Pseudo-2D mean"]
            if has_std:
                titles.append("Pseudo-2D std")
            if rho_true is not None:
                titles.append("True model")

            finite_vals = [sec_rho]
            if rho_true is not None:
                finite_vals.append(rho_true)
            finite = np.concatenate([v[np.isfinite(v)] for v in finite_vals if np.asarray(v).size])
            zmin_c = float(np.nanmin(finite)) if finite.size else None
            zmax_c = float(np.nanmax(finite)) if finite.size else None

            fig = make_subplots(
                rows=1, cols=ncols, subplot_titles=titles,
                horizontal_spacing=0.06, shared_xaxes="all", shared_yaxes="all",
            )

            col = 1
            fig.add_trace(go.Heatmap(
                x=sec_x, y=sec_z, z=sec_rho,
                colorscale="Viridis", zmin=zmin_c, zmax=zmax_c, showscale=False,
            ), row=1, col=col)
            col += 1

            if has_std:
                fig.add_trace(go.Heatmap(
                    x=sec_x, y=sec_z, z=sec_std,
                    colorscale="Hot", showscale=True,
                    colorbar=dict(title="Std", x=col / (ncols + 0.5), len=0.92),
                ), row=1, col=col)
                col += 1

            if rho_true is not None:
                fig.add_trace(go.Heatmap(
                    x=x_true, y=z_true, z=rho_true,
                    colorscale="Viridis", zmin=zmin_c, zmax=zmax_c,
                    showscale=True, colorbar=dict(title="Ohm.m", x=1.02, len=0.92),
                ), row=1, col=col)

            for c in range(1, ncols + 1):
                fig.update_xaxes(title_text="x (m)", row=1, col=c)
                fig.update_yaxes(title_text="z (m)", autorange="reversed", row=1, col=c)
            fig.update_xaxes(matches="x")
            fig.update_yaxes(matches="y")
            fig.update_layout(
                title="Pseudo-2D section overview",
                height=620, margin=dict(t=70, b=50, l=60, r=100),
            )
            display(fig)
        except Exception as exc:
            display(ipw.HTML(f"Section plot error: {exc}"))


def update_tx_diag(*_):
    data = state.get("run_data")
    if data is None:
        return
    with tx_diag_out:
        clear_output(wait=True)
        try:
            tx_id = int(tx_select.value)
            if tx_id < 0:
                display(ipw.HTML("No Tx selected."))
                return

            tx_ids = data.get("tx_ids", np.array([])).astype(int)
            idx = int(np.where(tx_ids == tx_id)[0][0]) if tx_id in tx_ids else None
            if idx is None:
                display(ipw.HTML(f"Tx {tx_id} not found in run data."))
                return

            rho_mean = data["rho_layers"][idx]
            rho_std = data.get("rho_layers_std", np.zeros_like(data["rho_layers"]))[idx]
            thk_mean = data["thickness_layers"][idx]

            valid = np.isfinite(rho_mean)
            rho_m = rho_mean[valid]
            rho_s = rho_std[valid]
            thk_valid = np.isfinite(thk_mean)
            thk_m = thk_mean[thk_valid]

            if rho_m.size == 0:
                display(ipw.HTML("No valid layers for this Tx."))
                return

            depths = np.zeros(rho_m.size + 1, dtype=float)
            if thk_m.size > 0:
                depths[1:thk_m.size + 1] = np.cumsum(thk_m)
                depths[thk_m.size + 1:] = depths[thk_m.size]
            z_centers = 0.5 * (depths[:-1] + depths[1:])
            z_centers[-1] = depths[-2] + 0.5 * (thk_m[-1] if thk_m.size else 10.0)

            misfit_mean = float(data.get("misfit_mean", data["misfit"])[idx])
            misfit_std = float(data.get("misfit_std", np.zeros(1))[min(idx, data.get("misfit_std", np.zeros(1)).size - 1)])

            # Resistivity vs depth with horizontal uncertainty (error bars only).
            err_factor = 1.0 if uncertainty_select.value == "std" else 1.645
            rho_err = err_factor * np.asarray(rho_s, dtype=float)
            rho_err[~np.isfinite(rho_err)] = 0.0

            fig = go.Figure()
            fig.add_trace(
                go.Scatter(
                    x=rho_m,
                    y=z_centers,
                    mode="lines+markers",
                    name="Mean rho",
                    line=dict(color="blue", width=2),
                    marker=dict(size=6),
                    error_x=dict(type="data", array=rho_err, visible=True, color="red"),
                )
            )
            fig.update_xaxes(title_text="Resistivity (Ohm.m)")
            fig.update_yaxes(title_text="Relative depth (m)", autorange="reversed")

            fig.update_layout(
                title=f"Tx {tx_id} resistivity vs depth (misfit: {misfit_mean:.4e} +/- {misfit_std:.4e}; {uncertainty_select.value})",
                height=430, margin=dict(t=60, b=50, l=60, r=20),
                legend=dict(orientation="h", y=-0.2),
            )
            display(fig)
        except Exception as exc:
            display(ipw.HTML(f"Tx diagnostic error: {exc}"))


def update_data_compare(*_):
    data = state.get("run_data")
    if data is None:
        return
    with data_compare_out:
        clear_output(wait=True)
        try:
            tx_id = int(tx_select.value)
            if tx_id < 0:
                display(ipw.HTML("No Tx selected."))
                return
            fidx = int(freq_select.value)

            pred_hxh_key = f"pred_hxh_mean_tx{tx_id}"
            pred_hxhz_key = f"pred_hxhz_mean_tx{tx_id}"
            obs_hxh_key = f"obs_hxh_tx{tx_id}"
            obs_hxhz_key = f"obs_hxhz_tx{tx_id}"
            std_hxh_key = f"pred_hxh_std_tx{tx_id}"
            std_hxhz_key = f"pred_hxhz_std_tx{tx_id}"
            rx_key = f"rx_x_tx{tx_id}"

            if pred_hxh_key not in data:
                display(ipw.HTML(f"No predicted data for Tx {tx_id}. Re-export from 05 notebook."))
                return

            pred_hxh = data[pred_hxh_key]
            pred_hxhz = data[pred_hxhz_key]
            obs_hxh = data.get(obs_hxh_key)
            obs_hxhz = data.get(obs_hxhz_key)
            std_hxh = data.get(std_hxh_key)
            std_hxhz = data.get(std_hxhz_key)
            rx_x = data.get(rx_key)

            fidx = max(0, min(fidx, pred_hxh.shape[0] - 1))
            x_axis = rx_x if rx_x is not None else np.arange(pred_hxh.shape[1])
            x_label = "Rx x (m)" if rx_x is not None else "Local rx index"

            freqs_key = f"freqs_tx{tx_id}"
            freq_val = float(data[freqs_key][fidx]) if freqs_key in data else fidx

            fig = make_subplots(rows=1, cols=2, subplot_titles=[
                f"Hx-Hx phase (f={freq_val:g} Hz)",
                f"Hx-Hz phase (f={freq_val:g} Hz)",
            ], horizontal_spacing=0.10)

            cos_pred_hxh = np.cos(pred_hxh[fidx])
            cos_pred_hxhz = np.cos(pred_hxhz[fidx])

            fig.add_trace(go.Scatter(
                x=x_axis, y=cos_pred_hxh, mode="lines+markers", name="Pred mean",
                marker=dict(size=5),
            ), row=1, col=1)

            if std_hxh is not None and np.any(std_hxh > 0):
                lo = np.cos(pred_hxh[fidx] + std_hxh[fidx])
                hi = np.cos(pred_hxh[fidx] - std_hxh[fidx])
                fig.add_trace(go.Scatter(
                    x=np.concatenate([x_axis, x_axis[::-1]]),
                    y=np.concatenate([hi, lo[::-1]]),
                    fill="toself", fillcolor="rgba(0,100,255,0.15)",
                    line=dict(width=0), name="Pred +/- std", showlegend=True,
                ), row=1, col=1)

            if obs_hxh is not None:
                fig.add_trace(go.Scatter(
                    x=x_axis, y=np.cos(obs_hxh[fidx]),
                    mode="lines+markers", name="Observed",
                    marker=dict(size=5, color="red"),
                    line=dict(color="red"),
                ), row=1, col=1)

            fig.add_trace(go.Scatter(
                x=x_axis, y=cos_pred_hxhz, mode="lines+markers", name="Pred mean",
                marker=dict(size=5), showlegend=False,
            ), row=1, col=2)

            if std_hxhz is not None and np.any(std_hxhz > 0):
                lo2 = np.cos(pred_hxhz[fidx] + std_hxhz[fidx])
                hi2 = np.cos(pred_hxhz[fidx] - std_hxhz[fidx])
                fig.add_trace(go.Scatter(
                    x=np.concatenate([x_axis, x_axis[::-1]]),
                    y=np.concatenate([hi2, lo2[::-1]]),
                    fill="toself", fillcolor="rgba(0,100,255,0.15)",
                    line=dict(width=0), name="Pred +/- std", showlegend=False,
                ), row=1, col=2)

            if obs_hxhz is not None:
                fig.add_trace(go.Scatter(
                    x=x_axis, y=np.cos(obs_hxhz[fidx]),
                    mode="lines+markers", name="Observed",
                    marker=dict(size=5, color="red"),
                    line=dict(color="red"), showlegend=False,
                ), row=1, col=2)

            for c in (1, 2):
                fig.update_xaxes(title_text=x_label, row=1, col=c)
                fig.update_yaxes(title_text="cos(phase)", range=[-1.05, 1.05], row=1, col=c)

            fig.update_layout(
                title=f"Tx {tx_id}: Modelled vs observed cosine-domain phases",
                height=460, margin=dict(t=60, b=50, l=60, r=20),
                legend=dict(orientation="h", y=-0.18),
            )
            display(fig)

            # Misfit overview across all Tx
            tx_ids = data.get("tx_ids", np.array([])).astype(int)
            tx_x_arr = data.get("tx_x", np.arange(tx_ids.size))
            misfit_mean = data.get("misfit_mean", data.get("misfit", np.array([])))
            misfit_std = data.get("misfit_std")

            if tx_ids.size > 0 and misfit_mean.size > 0:
                fig2 = go.Figure()
                fig2.add_trace(go.Scatter(
                    x=tx_x_arr, y=misfit_mean, mode="lines+markers", name="Misfit mean",
                    error_y=dict(
                        type="data",
                        array=misfit_std.tolist() if misfit_std is not None else None,
                        visible=misfit_std is not None and np.any(misfit_std > 0),
                    ),
                ))
                fig2.update_layout(
                    title="Misfit vs Tx position (mean +/- std across ensemble runs)",
                    xaxis_title="Tx x (m)", yaxis_title="Objective",
                    height=280, margin=dict(t=40, b=40, l=60, r=20),
                )
                display(fig2)

        except Exception as exc:
            display(ipw.HTML(f"Data compare error: {exc}"))


# -------------------- Per-Tx resistivity diagnostics (error bars only) --------------------
def update_tx_diag(*_):
    data = state.get("run_data")
    if data is None:
        return
    with tx_diag_out:
        clear_output(wait=True)
        try:
            tx_id = int(tx_select.value)
            if tx_id < 0:
                display(ipw.HTML("No Tx selected."))
                return

            tx_ids = data.get("tx_ids", np.array([])).astype(int)
            idx = int(np.where(tx_ids == tx_id)[0][0]) if tx_id in tx_ids else None
            if idx is None:
                display(ipw.HTML(f"Tx {tx_id} not found in run data."))
                return

            rho_mean = data["rho_layers"][idx]
            rho_std = data.get("rho_layers_std", np.zeros_like(data["rho_layers"]))[idx]
            thk_mean = data["thickness_layers"][idx]

            valid = np.isfinite(rho_mean)
            rho_m = np.asarray(rho_mean[valid], dtype=float)
            rho_s = np.asarray(rho_std[valid], dtype=float)
            thk_valid = np.isfinite(thk_mean)
            thk_m = np.asarray(thk_mean[thk_valid], dtype=float)

            if rho_m.size == 0:
                display(ipw.HTML("No valid layers for this Tx."))
                return

            depths = np.zeros(rho_m.size + 1, dtype=float)
            if thk_m.size > 0:
                depths[1:thk_m.size + 1] = np.cumsum(thk_m)
                depths[thk_m.size + 1:] = depths[thk_m.size]
            z_centers = 0.5 * (depths[:-1] + depths[1:])
            z_centers[-1] = depths[-2] + 0.5 * (thk_m[-1] if thk_m.size else 10.0)

            misfit_mean = float(data.get("misfit_mean", data.get("misfit", 0.0))[idx])
            misfit_std = float(data.get("misfit_std", np.zeros(1))[min(idx, data.get("misfit_std", np.zeros(1)).size - 1)])

            unc_mode = uncertainty_select.value
            err_factor = 1.0 if unc_mode == "std" else 1.645
            rho_err = np.maximum(0.0, err_factor * rho_s)
            rho_err[~np.isfinite(rho_err)] = 0.0

            fig = go.Figure()
            fig.add_trace(
                go.Scatter(
                    x=rho_m,
                    y=z_centers,
                    mode="lines+markers",
                    name="Mean rho",
                    line=dict(color="blue", width=2),
                    marker=dict(size=6),
                    error_x=dict(type="data", array=rho_err, visible=True, color="red"),
                )
            )
            fig.update_xaxes(title_text="Resistivity (Ohm.m)")
            fig.update_yaxes(title_text="Relative depth (m)", autorange="reversed")

            fig.update_layout(
                title=f"Tx {tx_id} resistivity vs depth (misfit: {misfit_mean:.4e} +/- {misfit_std:.4e}; {unc_mode})",
                height=430,
                margin=dict(t=60, b=50, l=60, r=20),
                legend=dict(orientation="h", y=-0.2),
            )
            display(fig)
        except Exception as exc:
            display(ipw.HTML(f"Tx diagnostic error: {exc}"))

# -------------------- Standard data-compare (04_results style) --------------------
def refresh_data_compare_controls():
    base = state.get('real_result') or state.get('syn_result')
    if base is None:
        comp_freq.options = [('n/a', 0)]
        comp_freq.value = 0
        tx_select.options = [('n/a', 0)]
        tx_select.value = 0
        rx_local_select.min = 0
        rx_local_select.max = 0
        rx_local_select.value = 0
        trace_idx.max = 0
        trace_idx.value = 0
        return

    geo = base['geometry']
    comp_key = component_select.value if component_select.value in ('Hx', 'Hz') else 'Hx'
    comp_data = base.get(comp_key, base.get('Hx', {}))
    freqs = np.asarray(comp_data.get('freqs', []), dtype=float)

    if freqs.size == 0:
        comp_freq.options = [('n/a', 0)]
        comp_freq.value = 0
        return

    comp_freq.options = [(f'{f:g} Hz', i) for i, f in enumerate(freqs)]
    comp_freq.value = int(min(comp_freq.value if comp_freq.value is not None else 0, freqs.size - 1))

    tx_vals = np.unique(np.asarray(geo.get('tx_idx_per_trace', []), dtype=int))
    if tx_vals.size:
        tx_select.options = [(f'Tx {int(v)}', int(v)) for v in tx_vals]
        tx_select.value = int(tx_vals[0])
    else:
        tx_select.options = [('n/a', 0)]
        tx_select.value = 0

    rx_local = np.asarray(geo.get('rx_local_idx_per_trace', geo.get('rx_idx_per_trace', [])), dtype=int)
    if rx_local.size:
        rx_local_select.min = int(np.nanmin(rx_local))
        rx_local_select.max = int(np.nanmax(rx_local))
        rx_local_select.value = int(rx_local_select.min)
    else:
        rx_local_select.min = 0
        rx_local_select.max = 0
        rx_local_select.value = 0

    ntr = int(np.asarray(geo.get('tx_idx_per_trace', [])).size)
    trace_idx.max = max(0, ntr - 1)
    trace_idx.value = int(min(trace_idx.value, trace_idx.max))


def _same_geometry(real, syn):
    if real is None or syn is None:
        return False
    rg = real['geometry']
    sg = syn['geometry']
    return (
        np.asarray(rg['tx_idx_per_trace']).shape == np.asarray(sg['tx_idx_per_trace']).shape
        and np.asarray(rg['rx_idx_per_trace']).shape == np.asarray(sg['rx_idx_per_trace']).shape
    )


def _phase_deg(arr):
    return np.rad2deg(np.asarray(arr, dtype=float))


def update_data_compare_plot(*_):
    real = state.get('real_result')
    syn = state.get('syn_result')

    with data_compare_out:
        clear_output(wait=True)
        if real is None and syn is None:
            return
        if real is None:
            display(ipw.HTML('Synthetic loaded. Load real data to compare.'))
            return

        comp = component_select.value
        metric = metric_select.value
        fidx = int(comp_freq.value) if comp_freq.value is not None else 0
        tx_id = int(tx_select.value) if tx_select.value is not None else 0
        rx_local_target = int(rx_local_select.value)
        tr_pick = int(trace_idx.value)

        comp_data = real.get(comp)
        if comp_data is None:
            display(ipw.HTML(f'Missing {comp} data in real result.'))
            return

        geo = real['geometry']
        tx_arr = np.asarray(geo.get('tx_idx_per_trace', []), dtype=int)
        rx_arr = np.asarray(geo.get('rx_idx_per_trace', []), dtype=int)
        rx_local = np.asarray(geo.get('rx_local_idx_per_trace', rx_arr), dtype=int)
        freqs = np.asarray(comp_data.get('freqs', []), dtype=float)

        if tx_arr.size == 0 or rx_local.size == 0:
            display(ipw.HTML('Loaded data has no trace geometry to plot.'))
            return
        if freqs.size == 0:
            display(ipw.HTML('Loaded data has no frequencies to plot.'))
            return

        fidx = int(max(0, min(fidx, freqs.size - 1)))

        have_syn = syn is not None and _same_geometry(real, syn) and comp in syn
        syn_comp = syn.get(comp) if have_syn else None

        fig = go.Figure()
        title = ''

        if metric == 'amp_vs_rx':
            idx = np.where(tx_arr == tx_id)[0]
            if idx.size == 0:
                display(ipw.HTML('No traces for selected tx.'))
                return
            x = rx_local[idx]
            order = np.argsort(x)
            x = x[order]
            y_real = np.asarray(comp_data['amp_mean'][fidx, idx], dtype=float)[order]
            fig.add_trace(go.Scatter(x=x, y=y_real, mode='lines+markers', name='Real'))
            if syn_comp is not None:
                y_syn = np.asarray(syn_comp['amp_mean'][fidx, idx], dtype=float)[order]
                fig.add_trace(go.Scatter(x=x, y=y_syn, mode='lines+markers', name='Synthetic'))
            title = f'{comp} amplitude vs local rx (Tx {tx_id}, f={freqs[fidx]:g} Hz)'
            fig.update_xaxes(title_text='Local rx index')
            fig.update_yaxes(title_text='Amplitude')

        elif metric == 'phase_vs_rx_deg':
            idx = np.where(tx_arr == tx_id)[0]
            if idx.size == 0:
                display(ipw.HTML('No traces for selected tx.'))
                return
            x = rx_local[idx]
            order = np.argsort(x)
            x = x[order]
            y_real = _phase_deg(comp_data['phi_mean_rad'][fidx, idx])[order]
            fig.add_trace(go.Scatter(x=x, y=y_real, mode='lines+markers', name='Real'))
            if syn_comp is not None:
                y_syn = _phase_deg(syn_comp['phi_mean_rad'][fidx, idx])[order]
                fig.add_trace(go.Scatter(x=x, y=y_syn, mode='lines+markers', name='Synthetic'))
            title = f'{comp} phase vs local rx (Tx {tx_id}, f={freqs[fidx]:g} Hz)'
            fig.update_xaxes(title_text='Local rx index')
            fig.update_yaxes(title_text='Phase (deg)')

        elif metric == 'amp_vs_tx':
            tx_vals = np.unique(tx_arr)
            xs, y_real, y_syn = [], [], []
            for t in tx_vals:
                cand = np.where((tx_arr == int(t)) & (rx_local == rx_local_target))[0]
                if cand.size == 0:
                    continue
                i = int(cand[0])
                xs.append(int(t))
                y_real.append(float(comp_data['amp_mean'][fidx, i]))
                if syn_comp is not None:
                    y_syn.append(float(syn_comp['amp_mean'][fidx, i]))
            if not xs:
                display(ipw.HTML('No traces for selected local rx across tx.'))
                return
            fig.add_trace(go.Scatter(x=xs, y=y_real, mode='lines+markers', name='Real'))
            if syn_comp is not None:
                fig.add_trace(go.Scatter(x=xs, y=y_syn, mode='lines+markers', name='Synthetic'))
            title = f'{comp} amplitude vs tx (local rx {rx_local_target}, f={freqs[fidx]:g} Hz)'
            fig.update_xaxes(title_text='Tx index')
            fig.update_yaxes(title_text='Amplitude')

        elif metric == 'phase_vs_tx_deg':
            tx_vals = np.unique(tx_arr)
            xs, y_real, y_syn = [], [], []
            for t in tx_vals:
                cand = np.where((tx_arr == int(t)) & (rx_local == rx_local_target))[0]
                if cand.size == 0:
                    continue
                i = int(cand[0])
                xs.append(int(t))
                y_real.append(float(_phase_deg(comp_data['phi_mean_rad'][fidx, i])))
                if syn_comp is not None:
                    y_syn.append(float(_phase_deg(syn_comp['phi_mean_rad'][fidx, i])))
            if not xs:
                display(ipw.HTML('No traces for selected local rx across tx.'))
                return
            fig.add_trace(go.Scatter(x=xs, y=y_real, mode='lines+markers', name='Real'))
            if syn_comp is not None:
                fig.add_trace(go.Scatter(x=xs, y=y_syn, mode='lines+markers', name='Synthetic'))
            title = f'{comp} phase vs tx (local rx {rx_local_target}, f={freqs[fidx]:g} Hz)'
            fig.update_xaxes(title_text='Tx index')
            fig.update_yaxes(title_text='Phase (deg)')

        elif metric == 'amp_vs_freq':
            tr_pick = max(0, min(tr_pick, tx_arr.size - 1))
            y_real = np.asarray(comp_data['amp_mean'][:, tr_pick], dtype=float)
            fig.add_trace(go.Scatter(x=freqs, y=y_real, mode='lines+markers', name='Real'))
            if syn_comp is not None:
                y_syn = np.asarray(syn_comp['amp_mean'][:, tr_pick], dtype=float)
                fig.add_trace(go.Scatter(x=freqs, y=y_syn, mode='lines+markers', name='Synthetic'))
            title = f'{comp} amplitude vs frequency (trace {tr_pick}, tx={tx_arr[tr_pick]}, local_rx={rx_local[tr_pick]})'
            fig.update_xaxes(title_text='Frequency (Hz)', type='log')
            fig.update_yaxes(title_text='Amplitude')

        elif metric == 'phase_vs_freq_deg':
            tr_pick = max(0, min(tr_pick, tx_arr.size - 1))
            y_real = _phase_deg(comp_data['phi_mean_rad'][:, tr_pick])
            fig.add_trace(go.Scatter(x=freqs, y=y_real, mode='lines+markers', name='Real'))
            if syn_comp is not None:
                y_syn = _phase_deg(syn_comp['phi_mean_rad'][:, tr_pick])
                fig.add_trace(go.Scatter(x=freqs, y=y_syn, mode='lines+markers', name='Synthetic'))
            title = f'{comp} phase vs frequency (trace {tr_pick}, tx={tx_arr[tr_pick]}, local_rx={rx_local[tr_pick]})'
            fig.update_xaxes(title_text='Frequency (Hz)', type='log')
            fig.update_yaxes(title_text='Phase (deg)')

        else:
            display(ipw.HTML('Unknown plotting mode.'))
            return

        y_all = []
        for tr in fig.data:
            if getattr(tr, 'y', None) is not None:
                y_all.append(np.asarray(tr.y, dtype=float))
        y_vals = np.concatenate(y_all) if y_all else np.array([])
        y_vals = y_vals[np.isfinite(y_vals)] if y_vals.size else y_vals

        fig.update_layout(
            title=title,
            height=420,
            margin=dict(t=60, b=50, l=60, r=20),
            legend=dict(orientation='h', y=-0.2),
            xaxis_fixedrange=False,
            yaxis_fixedrange=False,
        )
        fig.update_xaxes(autorange=True)

        if metric in ('phase_vs_rx_deg', 'phase_vs_tx_deg', 'phase_vs_freq_deg'):
            fig.update_yaxes(range=[-180.0, 180.0])
        else:
            if y_vals.size:
                ymax = float(np.nanmax(y_vals))
                if not np.isfinite(ymax):
                    ymax = 1.0
                upper = max(1.0, ymax * 2.0)
                fig.update_yaxes(range=[0.0, upper])
            else:
                fig.update_yaxes(range=[0.0, 1.0])

        display(fig)


def update_data_compare(*_):
    update_data_compare_plot(*_)


def on_load_real(_):
    try:
        if state.get('summary') is None:
            raise RuntimeError('Load a 1D inversion run first.')

        freqs = parse_freqs(freqs_input.value)
        hx_path, hz_path = get_real_data_paths()
        result = compute_amp_phase_for_fd_outputs(hx_path, hz_path, freqs=freqs, n_pairs=3)

        phase_comp_deg = float(state['summary'].get('phase_comp_deg', 0.0))
        shift_rad = np.deg2rad(phase_comp_deg)

        result['Hx']['phi_mean_rad'] = wrap_phase_rad(result['Hx']['phi_mean_rad'] - shift_rad)
        result['Hz']['phi_mean_rad'] = wrap_phase_rad(result['Hz']['phi_mean_rad'] - shift_rad)

        state['real_result'] = result
        state['syn_result'] = None

        push_message('Real data loaded (phase compensated for plotting).')
        refresh_data_compare_controls()
        update_data_compare_plot()
    except Exception as exc:
        push_message(f'Load real failed: {exc}')
        push_message(traceback.format_exc())


def on_generate_syn(_):
    try:
        data = state.get('run_data')
        summary = state.get('summary')
        real = state.get('real_result')
        if data is None or summary is None:
            raise RuntimeError('Load a 1D inversion run first.')
        if real is None:
            raise RuntimeError('Load real data first (needed for geometry).')

        n_layers = int(summary['n_layers'])
        tilt_deg = float(summary.get('tilt_deg', 0.0))
        ab_hxh = int(summary.get('ab_hxh', 44))
        ab_hxhz = int(summary.get('ab_hxhz', 46))

        rho_layers = np.asarray(data['rho_layers'], dtype=float)
        thickness_layers = np.asarray(data['thickness_layers'], dtype=float)
        tx_ids = np.asarray(data['tx_ids'], dtype=int)

        geom = real['geometry']
        tx_idx_per_trace = np.asarray(geom['tx_idx_per_trace'], dtype=int)
        rx_x = np.asarray(geom['rx_x'], dtype=float)
        rx_z = np.asarray(geom['rx_z'], dtype=float)
        src_x = np.asarray(geom['src_x'], dtype=float)
        src_z = np.asarray(geom['src_z'], dtype=float)

        freqs = np.asarray(real['Hx']['freqs'], dtype=float)
        nfreq = int(freqs.size)
        ntrace = int(real['Hx']['amp_mean'].shape[1])

        syn_amp_hxh = np.full((nfreq, ntrace), np.nan, dtype=float)
        syn_phi_hxh = np.full((nfreq, ntrace), np.nan, dtype=float)
        syn_amp_hxhz = np.full((nfreq, ntrace), np.nan, dtype=float)
        syn_phi_hxhz = np.full((nfreq, ntrace), np.nan, dtype=float)

        for tx_id in tx_ids:
            idx = int(np.where(tx_ids == tx_id)[0][0])
            tr_idx = np.where(tx_idx_per_trace == int(tx_id))[0]
            if tr_idx.size == 0:
                continue

            rho = np.asarray(rho_layers[idx, :n_layers], dtype=float)
            thk = np.asarray(thickness_layers[idx, : n_layers - 1], dtype=float)

            tx_x_val = float(src_x[tr_idx[0]])
            tx_z_val = float(src_z[tr_idx[0]])
            off_x = rx_x[tr_idx] - tx_x_val
            off_z = rx_z[tr_idx] - tx_z_val

            amp_hxh, phi_hxh, amp_hxhz, phi_hxhz = forward_empymod_1d_layered_amp_phase(
                rho=rho,
                thickness=thk,
                freqs=freqs,
                off_x=off_x,
                off_z=off_z,
                tilt_deg=tilt_deg,
                ab_hxh=ab_hxh,
                ab_hxhz=ab_hxhz,
            )

            syn_amp_hxh[:, tr_idx] = amp_hxh
            syn_phi_hxh[:, tr_idx] = phi_hxh
            syn_amp_hxhz[:, tr_idx] = amp_hxhz
            syn_phi_hxhz[:, tr_idx] = phi_hxhz

        syn_result = {
            'freqs': freqs,
            'geometry': geom,
            'Hx': {
                'nt': int(real['Hx'].get('nt', 0)),
                'ntrace': ntrace,
                'dt': float(real['Hx'].get('dt', 0.0)),
                'freqs': freqs,
                'amp_mean': syn_amp_hxh,
                'amp_std': np.zeros_like(syn_amp_hxh),
                'phi_mean_rad': syn_phi_hxh,
                'phi_std_rad': np.zeros_like(syn_phi_hxh),
            },
            'Hz': {
                'nt': int(real['Hz'].get('nt', 0)),
                'ntrace': ntrace,
                'dt': float(real['Hz'].get('dt', 0.0)),
                'freqs': freqs,
                'amp_mean': syn_amp_hxhz,
                'amp_std': np.zeros_like(syn_amp_hxhz),
                'phi_mean_rad': syn_phi_hxhz,
                'phi_std_rad': np.zeros_like(syn_phi_hxhz),
            },
        }

        state['syn_result'] = syn_result
        push_message('Synthetics generated from inverted empymod models.')
        refresh_data_compare_controls()
        update_data_compare_plot()
    except Exception as exc:
        push_message(f'Generate synthetics failed: {exc}')
        push_message(traceback.format_exc())


def update_rho_slices_plot(*_):
    sec_x = state.get('section_x')
    sec_z = state.get('section_z')
    sec_rho = state.get('section_rho')
    true_x = state.get('true_x')
    true_z = state.get('true_z')
    true_rho = state.get('true_rho')

    if sec_x is None or sec_z is None or sec_rho is None:
        return

    with rho_z_slice_out:
        clear_output(wait=True)
        with rho_x_slice_out:
            pass

    x_pick = float(x_slice_input.value)
    z_pick = float(z_slice_input.value)

    ix = int(np.nanargmin(np.abs(np.asarray(sec_x, dtype=float) - x_pick)))
    mean_rho_at_x = np.asarray(sec_rho, dtype=float)[:, ix]

    ix_true = None
    true_rho_at_x = None
    if true_x is not None and true_rho is not None:
        ix_true = int(np.nanargmin(np.abs(np.asarray(true_x, dtype=float) - x_pick)))
        true_rho_at_x = np.asarray(true_rho, dtype=float)[:, ix_true]

    with rho_z_slice_out:
        clear_output(wait=True)
        fig_z = go.Figure()
        fig_z.add_trace(go.Scatter(x=mean_rho_at_x, y=np.asarray(sec_z, dtype=float), mode='lines', name='Mean'))
        if true_rho_at_x is not None and true_z is not None:
            fig_z.add_trace(go.Scatter(x=true_rho_at_x, y=np.asarray(true_z, dtype=float), mode='lines', name='True'))
        fig_z.update_xaxes(title_text='Resistivity (Ohm.m)')
        fig_z.update_yaxes(title_text='z (m)', autorange='reversed')
        fig_z.update_layout(height=310, margin=dict(t=40, b=30, l=60, r=20), legend=dict(orientation='h', y=-0.2))
        fig_z.update_layout(title=f'x slice: {x_pick:g} m')
        display(fig_z)

    iz = int(np.nanargmin(np.abs(np.asarray(sec_z, dtype=float) - z_pick)))
    mean_rho_at_z = np.asarray(sec_rho, dtype=float)[iz, :]

    iz_true = None
    true_rho_at_z = None
    if true_z is not None and true_rho is not None:
        iz_true = int(np.nanargmin(np.abs(np.asarray(true_z, dtype=float) - z_pick)))
        true_rho_at_z = np.asarray(true_rho, dtype=float)[iz_true, :]

    with rho_x_slice_out:
        clear_output(wait=True)
        fig_x = go.Figure()
        fig_x.add_trace(go.Scatter(x=np.asarray(sec_x, dtype=float), y=mean_rho_at_z, mode='lines', name='Mean'))
        if true_rho_at_z is not None and true_x is not None:
            fig_x.add_trace(go.Scatter(x=np.asarray(true_x, dtype=float), y=true_rho_at_z, mode='lines', name='True'))
        fig_x.update_xaxes(title_text='x (m)')
        fig_x.update_yaxes(title_text='Resistivity (Ohm.m)')
        fig_x.update_layout(height=310, margin=dict(t=40, b=30, l=60, r=20), legend=dict(orientation='h', y=-0.2))
        fig_x.update_layout(title=f'z slice: {z_pick:g} m')
        display(fig_x)


def update_section_plot():
    data = state.get('run_data')
    if data is None:
        return
    with section_out:
        clear_output(wait=True)
        try:
            sec_x = data.get('section_x')
            sec_z = data.get('section_z')
            sec_rho = data.get('section_rho')
            sec_std = data.get('section_rho_std')

            force_rebuild = bool(state.get('force_rebuild_section', False))
            if force_rebuild or sec_x is None or sec_z is None or sec_rho is None:
                try:
                    sec_x, sec_z, sec_rho, sec_std = _rebuild_pseudosection_from_saved_layers(
                        data,
                        state.get('summary') or {},
                        background_rho_override=float(bg_rho_input.value),
                    )
                    data['section_x'] = sec_x
                    data['section_z'] = sec_z
                    data['section_rho'] = sec_rho
                    data['section_rho_std'] = sec_std
                    state['force_rebuild_section'] = False
                except Exception as exc:
                    display(ipw.HTML(f'Could not rebuild pseudo-2D section from saved results: {exc}'))
                    return

            has_std = sec_std is not None and np.any(np.isfinite(np.asarray(sec_std, dtype=float)))

            rho_true = None
            x_true = None
            z_true = None
            if SG_TRUE_PATH.exists():
                try:
                    x_true, z_true, g_true = _read_rss_model(SG_TRUE_PATH)
                    rho_true = conductivity_to_resistivity(g_true)
                except Exception:
                    pass

            # Cache arrays for slice plotting.
            state['section_x'] = np.asarray(sec_x, dtype=float)
            state['section_z'] = np.asarray(sec_z, dtype=float)
            state['section_rho'] = np.asarray(sec_rho, dtype=float)
            state['section_rho_std'] = np.asarray(sec_std, dtype=float) if sec_std is not None else None
            state['true_x'] = np.asarray(x_true, dtype=float) if x_true is not None else None
            state['true_z'] = np.asarray(z_true, dtype=float) if z_true is not None else None
            state['true_rho'] = np.asarray(rho_true, dtype=float) if rho_true is not None else None

            # Keep slice picks in range.
            sec_x_min, sec_x_max = float(np.nanmin(sec_x)), float(np.nanmax(sec_x))
            sec_z_min, sec_z_max = float(np.nanmin(sec_z)), float(np.nanmax(sec_z))
            if not (sec_x_min <= float(x_slice_input.value) <= sec_x_max):
                x_slice_input.value = float(np.asarray(sec_x, dtype=float)[len(sec_x)//2])
            if not (sec_z_min <= float(z_slice_input.value) <= sec_z_max):
                z_slice_input.value = float(np.asarray(sec_z, dtype=float)[len(sec_z)//2])

            finite_vals = [np.asarray(sec_rho, dtype=float)]
            if rho_true is not None:
                finite_vals.append(np.asarray(rho_true, dtype=float))
            finite = np.concatenate([v[np.isfinite(v)] for v in finite_vals if np.asarray(v).size])
            zmin_c = float(np.nanmin(finite)) if finite.size else None
            zmax_c = float(np.nanmax(finite)) if finite.size else None

            fig_mean = go.Figure()
            fig_mean.add_trace(go.Heatmap(x=sec_x, y=sec_z, z=sec_rho, colorscale='Viridis', zmin=zmin_c, zmax=zmax_c))
            fig_mean.update_xaxes(title_text='x (m)')
            fig_mean.update_yaxes(title_text='z (m)', autorange='reversed')
            fig_mean.update_layout(title='Pseudo-2D mean', height=260, margin=dict(t=50, b=35, l=60, r=20))
            display(fig_mean)

            if has_std:
                finite_s = np.asarray(sec_std)[np.isfinite(sec_std)]
                zmin_s = float(np.nanmin(finite_s)) if finite_s.size else None
                zmax_s = float(np.nanmax(finite_s)) if finite_s.size else None
                fig_std = go.Figure()
                fig_std.add_trace(go.Heatmap(x=sec_x, y=sec_z, z=sec_std, colorscale='Hot', zmin=zmin_s, zmax=zmax_s))
                fig_std.update_xaxes(title_text='x (m)')
                fig_std.update_yaxes(title_text='z (m)', autorange='reversed')
                fig_std.update_layout(title='Pseudo-2D std', height=260, margin=dict(t=50, b=35, l=60, r=20))
                display(fig_std)

            if rho_true is not None and x_true is not None and z_true is not None:
                fig_true = go.Figure()
                fig_true.add_trace(go.Heatmap(x=x_true, y=z_true, z=rho_true, colorscale='Viridis', zmin=zmin_c, zmax=zmax_c))
                fig_true.update_xaxes(title_text='x (m)')
                fig_true.update_yaxes(title_text='z (m)', autorange='reversed')
                fig_true.update_layout(title='True model', height=260, margin=dict(t=50, b=35, l=60, r=20))
                display(fig_true)

            update_rho_slices_plot()
        except Exception as exc:
            display(ipw.HTML(f'Section plot error: {exc}'))

        

def on_export_segy(_):
    data = state.get("run_data")
    run_dir = state.get("run_dir")
    if data is None or run_dir is None:
        push_message("Load a run first.")
        return
    sec_rho = data.get("section_rho")
    sec_x = data.get("section_x")
    sec_z = data.get("section_z")
    if sec_rho is None:
        push_message("No section data in this run.")
        return
    template_path, template_src = _resolve_segy_template_path()
    if template_path is None:
        push_message("No SEG-Y template found.")
        export_status.value = '<span style="color:#aa0000">No SEG-Y template found.</span>'
        return
    try:
        out_dir = Path(run_dir) / "SEGY_output"
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / "empymod_1d_pseudo2d_mean.segy"
        info = write_resistivity_to_segy_from_template(
            template_segy_path=template_path,
            output_segy_path=out_path,
            resistivity_grid=np.asarray(sec_rho, dtype=float),
            x_model=np.asarray(sec_x, dtype=float),
            z_model=np.asarray(sec_z, dtype=float),
            method="linear",
        )
        push_message(f"SEGY export SUCCESS: {out_path}")
        export_status.value = f'<span style="color:#006400">SEGY exported: {out_path.name}</span>'
    except Exception as exc:
        push_message(f"SEGY export failed: {exc}")
        export_status.value = f'<span style="color:#aa0000">SEGY export failed: {exc}</span>'


def on_quit_gui(_):
    push_message("Shutting down 1D results GUI server...")
    try:
        if VOILA_PID_FILE.exists():
            pid_text = VOILA_PID_FILE.read_text().strip()
            if pid_text:
                pid = int(pid_text)
                if pid != os.getpid():
                    try:
                        os.kill(pid, signal.SIGTERM)
                    except Exception:
                        pass
    except Exception:
        pass
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip and getattr(ip, "kernel", None):
            ip.kernel.do_shutdown(restart=False)
    except Exception:
        pass
    try:
        os.kill(os.getpid(), signal.SIGTERM)
    except Exception:
        pass


load_btn.on_click(on_load_run)
refresh_runs_btn.on_click(lambda _: refresh_run_options())
rebuild_section_btn.on_click(on_rebuild_section)
refresh_tx_btn.on_click(lambda _: update_tx_diag())
uncertainty_select.observe(lambda c: update_tx_diag() if c.get("name") == "value" else None, names="value")
load_real_btn.on_click(on_load_real)
generate_syn_btn.on_click(on_generate_syn)
export_segy_btn.on_click(on_export_segy)
quit_btn.on_click(on_quit_gui)
tx_select.observe(lambda c: (update_tx_diag(), update_data_compare_plot()) if c.get("name") == "value" else None, names="value")
component_select.observe(lambda c: (refresh_data_compare_controls(), update_data_compare_plot()) if c.get("name") == "value" else None, names="value")
metric_select.observe(lambda c: update_data_compare_plot() if c.get("name") == "value" else None, names="value")
comp_freq.observe(lambda c: update_data_compare_plot() if c.get("name") == "value" else None, names="value")
rx_local_select.observe(lambda c: update_data_compare_plot() if c.get("name") == "value" else None, names="value")
trace_idx.observe(lambda c: update_data_compare_plot() if c.get("name") == "value" else None, names="value")
x_slice_input.observe(lambda c: update_rho_slices_plot() if c.get("name") == "value" else None, names="value")
z_slice_input.observe(lambda c: update_rho_slices_plot() if c.get("name") == "value" else None, names="value")

app = ipw.VBox([
    ipw.HTML("<h2>06_1d_empymod_results</h2>"),
    ipw.HBox([quit_btn], layout=ipw.Layout(justify_content="flex-end")),
    ipw.HTML("<p>Visualize and export 1D empymod ensemble inversion results.</p>"),
    ipw.HTML("<h3>1) Select run</h3>"),
    ipw.HBox([run_selector, load_btn, refresh_runs_btn]),
    ipw.HTML("<h3>2) Pseudo-2D section (mean, std, true model)</h3>"),
    section_out,
    ipw.HBox([bg_rho_input, rebuild_section_btn]),
    ipw.HBox([x_slice_input, z_slice_input]),
    ipw.HBox([rho_z_slice_out, rho_x_slice_out]),
    ipw.HTML("<h3>3) Per-Tx model diagnostics</h3>"),
    ipw.HBox([tx_select, refresh_tx_btn, uncertainty_select]),
    tx_diag_out,
    ipw.HTML("<h3>4) Modelled vs observed data (Real vs Synthetic)</h3>"),
    ipw.HBox([freqs_input, load_real_btn, generate_syn_btn]),
    ipw.HBox([component_select, metric_select, comp_freq, tx_select]),
    ipw.HBox([rx_local_select, trace_idx]),
    data_compare_out,
    ipw.HTML("<h3>5) Export</h3>"),
    ipw.HBox([export_segy_btn, export_status]),
    status_out,
])

display(app)
push_message("Ready. Select a OneDInversionRun from Results/Empymod1D and click Load.")

In [ ]:
# --- Amp+phase reconciliation patch for results notebook ---

FD_AMP_LINE_TO_POINT_SPREADING = True
FD_SPREADING_MIN_OFFSET_M = 1.0
FD_APPLY_TIME_DERIVATIVE = True
FD_DERIVATIVE_METHOD = "np_gradient"
FD_PHASE_CORRECTION_HXX_DEG = -180.0
FD_PHASE_CORRECTION_HXZ_DEG = 0.0


def _shared_amp_norm(a_hxh, a_hxhz):
    """Per-frequency shared amplitude normalization over Hxx and Hxz."""
    a1 = np.asarray(a_hxh, dtype=float)
    a2 = np.asarray(a_hxhz, dtype=float)
    if a1.ndim == 1:
        a1 = a1[None, :]
    if a2.ndim == 1:
        a2 = a2[None, :]
    if a1.size == 0 or a2.size == 0:
        return np.array([], dtype=float)

    n1 = np.nanmax(np.abs(a1), axis=1)
    n2 = np.nanmax(np.abs(a2), axis=1)
    n = np.nanmax(np.vstack([n1, n2]), axis=0)
    n = np.asarray(n, dtype=float)
    n[~np.isfinite(n)] = np.nan
    n[n <= 0.0] = np.nan
    return n


def _apply_fd_corrections_and_normalization(result):
    geo = result.get("geometry", {})
    tx_idx = np.asarray(geo.get("tx_idx_per_trace", []), dtype=int)
    rx_x = np.asarray(geo.get("rx_x", []), dtype=float)
    src_x = np.asarray(geo.get("src_x", []), dtype=float)

    amp_hx = np.asarray(result["Hx"]["amp_mean"], dtype=float)
    amp_hz = np.asarray(result["Hz"]["amp_mean"], dtype=float)
    phi_hx = np.asarray(result["Hx"]["phi_mean_rad"], dtype=float)
    phi_hz = np.asarray(result["Hz"]["phi_mean_rad"], dtype=float)

    for tx_id in np.unique(tx_idx):
        tr = np.where(tx_idx == int(tx_id))[0]
        if tr.size == 0:
            continue

        off_x = rx_x[tr] - src_x[tr]

        ahx = amp_hx[:, tr]
        ahz = amp_hz[:, tr]
        phx = phi_hx[:, tr]
        phz = phi_hz[:, tr]

        if FD_AMP_LINE_TO_POINT_SPREADING:
            spread = np.sqrt(np.maximum(np.abs(off_x), float(FD_SPREADING_MIN_OFFSET_M)))
            ahx = ahx / spread[None, :]
            ahz = ahz / spread[None, :]

        phx = wrap_phase_rad(phx + np.deg2rad(float(FD_PHASE_CORRECTION_HXX_DEG)))
        phz = wrap_phase_rad(phz + np.deg2rad(float(FD_PHASE_CORRECTION_HXZ_DEG)))

        nfd = np.asarray(_shared_amp_norm(ahx, ahz), dtype=float)
        if nfd.size == 0:
            nfd = np.ones(ahx.shape[0], dtype=float)
        bad = (~np.isfinite(nfd)) | (nfd <= 0.0)
        nfd[bad] = 1.0

        amp_hx[:, tr] = ahx / nfd[:, None]
        amp_hz[:, tr] = ahz / nfd[:, None]
        phi_hx[:, tr] = phx
        phi_hz[:, tr] = phz

    result["Hx"]["amp_mean"] = amp_hx
    result["Hz"]["amp_mean"] = amp_hz
    result["Hx"]["phi_mean_rad"] = phi_hx
    result["Hz"]["phi_mean_rad"] = phi_hz

    result["corrections"] = {
        "fd_phase_correction_hxx_deg": float(FD_PHASE_CORRECTION_HXX_DEG),
        "fd_phase_correction_hxz_deg": float(FD_PHASE_CORRECTION_HXZ_DEG),
        "fd_amp_line_to_point_spreading": bool(FD_AMP_LINE_TO_POINT_SPREADING),
        "fd_spreading_min_offset_m": float(FD_SPREADING_MIN_OFFSET_M),
        "fd_apply_time_derivative": bool(FD_APPLY_TIME_DERIVATIVE),
        "fd_derivative_method": str(FD_DERIVATIVE_METHOD),
        "amplitude_normalization": "shared_max_abs_hxx_hxz_per_frequency",
    }
    return result


_old_on_load_real = on_load_real
_old_on_generate_syn = on_generate_syn
_old_update_data_compare = update_data_compare


def on_load_real(_):
    try:
        if state.get("summary") is None:
            raise RuntimeError("Load a 1D inversion run first.")

        freqs = parse_freqs(freqs_input.value)
        hx_path, hz_path = get_real_data_paths()
        result = compute_amp_phase_for_fd_outputs(
            hx_path,
            hz_path,
            freqs=freqs,
            n_pairs=3,
            apply_time_derivative=FD_APPLY_TIME_DERIVATIVE,
            derivative_method=FD_DERIVATIVE_METHOD,
        )
        result = _apply_fd_corrections_and_normalization(result)

        state["real_result"] = result
        state["syn_result"] = None

        push_message("Real data loaded with FD time-derivative preprocessing (np.gradient) and FD->Empymod amp/phase corrections.")
        refresh_data_compare_controls()
        update_data_compare_plot()
    except Exception as exc:
        push_message(f"Load real failed: {exc}")
        push_message(traceback.format_exc())


def on_generate_syn(_):
    try:
        data = state.get("run_data")
        summary = state.get("summary")
        real = state.get("real_result")
        if data is None or summary is None:
            raise RuntimeError("Load a 1D inversion run first.")
        if real is None:
            raise RuntimeError("Load real data first (needed for geometry).")

        n_layers = int(summary["n_layers"])
        tilt_deg = float(summary.get("tilt_deg", 0.0))
        ab_hxh = int(summary.get("ab_hxh", 44))
        ab_hxhz = int(summary.get("ab_hxhz", 46))

        rho_layers = np.asarray(data["rho_layers"], dtype=float)
        thickness_layers = np.asarray(data["thickness_layers"], dtype=float)
        tx_ids = np.asarray(data["tx_ids"], dtype=int)

        geom = real["geometry"]
        tx_idx_per_trace = np.asarray(geom["tx_idx_per_trace"], dtype=int)
        rx_x = np.asarray(geom["rx_x"], dtype=float)
        rx_z = np.asarray(geom["rx_z"], dtype=float)
        src_x = np.asarray(geom["src_x"], dtype=float)
        src_z = np.asarray(geom["src_z"], dtype=float)

        freqs = np.asarray(real["Hx"]["freqs"], dtype=float)
        nfreq = int(freqs.size)
        ntrace = int(real["Hx"]["amp_mean"].shape[1])

        syn_amp_hxh = np.full((nfreq, ntrace), np.nan, dtype=float)
        syn_phi_hxh = np.full((nfreq, ntrace), np.nan, dtype=float)
        syn_amp_hxhz = np.full((nfreq, ntrace), np.nan, dtype=float)
        syn_phi_hxhz = np.full((nfreq, ntrace), np.nan, dtype=float)

        for tx_id in tx_ids:
            idx = int(np.where(tx_ids == tx_id)[0][0])
            tr_idx = np.where(tx_idx_per_trace == int(tx_id))[0]
            if tr_idx.size == 0:
                continue

            rho = np.asarray(rho_layers[idx, :n_layers], dtype=float)
            thk = np.asarray(thickness_layers[idx, : n_layers - 1], dtype=float)

            tx_x_val = float(src_x[tr_idx[0]])
            tx_z_val = float(src_z[tr_idx[0]])
            off_x = rx_x[tr_idx] - tx_x_val
            off_z = rx_z[tr_idx] - tx_z_val

            amp_hxh, phi_hxh, amp_hxhz, phi_hxhz = forward_empymod_1d_layered_amp_phase(
                rho=rho,
                thickness=thk,
                freqs=freqs,
                off_x=off_x,
                off_z=off_z,
                tilt_deg=tilt_deg,
                ab_hxh=ab_hxh,
                ab_hxhz=ab_hxhz,
            )

            nrm = np.asarray(_shared_amp_norm(amp_hxh, amp_hxhz), dtype=float)
            if nrm.size == 0:
                nrm = np.ones(amp_hxh.shape[0], dtype=float)
            bad = (~np.isfinite(nrm)) | (nrm <= 0.0)
            nrm[bad] = 1.0

            syn_amp_hxh[:, tr_idx] = amp_hxh / nrm[:, None]
            syn_phi_hxh[:, tr_idx] = phi_hxh
            syn_amp_hxhz[:, tr_idx] = amp_hxhz / nrm[:, None]
            syn_phi_hxhz[:, tr_idx] = phi_hxhz

        syn_result = {
            "freqs": freqs,
            "geometry": geom,
            "Hx": {
                "nt": int(real["Hx"].get("nt", 0)),
                "ntrace": ntrace,
                "dt": float(real["Hx"].get("dt", 0.0)),
                "freqs": freqs,
                "amp_mean": syn_amp_hxh,
                "amp_std": np.zeros_like(syn_amp_hxh),
                "phi_mean_rad": syn_phi_hxh,
                "phi_std_rad": np.zeros_like(syn_phi_hxh),
            },
            "Hz": {
                "nt": int(real["Hz"].get("nt", 0)),
                "ntrace": ntrace,
                "dt": float(real["Hz"].get("dt", 0.0)),
                "freqs": freqs,
                "amp_mean": syn_amp_hxhz,
                "amp_std": np.zeros_like(syn_amp_hxhz),
                "phi_mean_rad": syn_phi_hxhz,
                "phi_std_rad": np.zeros_like(syn_phi_hxhz),
            },
            "corrections": real.get("corrections", {}),
        }

        state["syn_result"] = syn_result
        push_message("Synthetics generated (shared Hxx+Hxz amplitude normalization per Tx).")
        refresh_data_compare_controls()
        update_data_compare_plot()
    except Exception as exc:
        push_message(f"Generate synthetics failed: {exc}")
        push_message(traceback.format_exc())


def update_data_compare(*_):
    data = state.get("run_data")
    if data is None:
        return
    with data_compare_out:
        clear_output(wait=True)
        try:
            tx_id = int(tx_select.value)
            if tx_id < 0:
                display(ipw.HTML("No Tx selected."))
                return
            fidx = int(freq_select.value)

            pred_hxh_key = f"pred_hxh_mean_tx{tx_id}"
            pred_hxhz_key = f"pred_hxhz_mean_tx{tx_id}"
            obs_hxh_key = f"obs_hxh_tx{tx_id}"
            obs_hxhz_key = f"obs_hxhz_tx{tx_id}"
            pred_ahxh_key = f"pred_amp_hxh_norm_tx{tx_id}"
            pred_ahxhz_key = f"pred_amp_hxhz_norm_tx{tx_id}"
            obs_ahxh_key = f"obs_amp_hxh_norm_tx{tx_id}"
            obs_ahxhz_key = f"obs_amp_hxhz_norm_tx{tx_id}"
            rx_key = f"rx_x_tx{tx_id}"

            if pred_hxh_key not in data:
                display(ipw.HTML(f"No predicted data for Tx {tx_id}. Re-export from 05 notebook."))
                return

            pred_hxh = np.asarray(data[pred_hxh_key], dtype=float)
            pred_hxhz = np.asarray(data[pred_hxhz_key], dtype=float)
            obs_hxh = np.asarray(data.get(obs_hxh_key), dtype=float) if obs_hxh_key in data else None
            obs_hxhz = np.asarray(data.get(obs_hxhz_key), dtype=float) if obs_hxhz_key in data else None

            pred_ahxh = np.asarray(data.get(pred_ahxh_key), dtype=float) if pred_ahxh_key in data else None
            pred_ahxhz = np.asarray(data.get(pred_ahxhz_key), dtype=float) if pred_ahxhz_key in data else None
            obs_ahxh = np.asarray(data.get(obs_ahxh_key), dtype=float) if obs_ahxh_key in data else None
            obs_ahxhz = np.asarray(data.get(obs_ahxhz_key), dtype=float) if obs_ahxhz_key in data else None

            fidx = max(0, min(fidx, pred_hxh.shape[0] - 1))
            rx_x = data.get(rx_key)
            x_axis = np.asarray(rx_x, dtype=float) if rx_x is not None else np.arange(pred_hxh.shape[1])
            x_label = "Rx x (m)" if rx_x is not None else "Local rx index"

            freqs_key = f"freqs_tx{tx_id}"
            freq_val = float(data[freqs_key][fidx]) if freqs_key in data else fidx

            fig = make_subplots(
                rows=2,
                cols=2,
                subplot_titles=[
                    f"Hx-Hx amplitude (norm, f={freq_val:g} Hz)",
                    f"Hx-Hz amplitude (norm, f={freq_val:g} Hz)",
                    "Hx-Hx phase (deg)",
                    "Hx-Hz phase (deg)",
                ],
                horizontal_spacing=0.10,
                vertical_spacing=0.12,
            )

            if pred_ahxh is not None:
                fig.add_trace(go.Scatter(x=x_axis, y=pred_ahxh[fidx], mode="lines+markers", name="Pred"), row=1, col=1)
            if obs_ahxh is not None:
                fig.add_trace(go.Scatter(x=x_axis, y=obs_ahxh[fidx], mode="lines+markers", name="Obs"), row=1, col=1)

            if pred_ahxhz is not None:
                fig.add_trace(go.Scatter(x=x_axis, y=pred_ahxhz[fidx], mode="lines+markers", name="Pred", showlegend=False), row=1, col=2)
            if obs_ahxhz is not None:
                fig.add_trace(go.Scatter(x=x_axis, y=obs_ahxhz[fidx], mode="lines+markers", name="Obs", showlegend=False), row=1, col=2)

            fig.add_trace(go.Scatter(x=x_axis, y=np.rad2deg(pred_hxh[fidx]), mode="lines+markers", name="Pred phase", showlegend=False), row=2, col=1)
            if obs_hxh is not None:
                fig.add_trace(go.Scatter(x=x_axis, y=np.rad2deg(obs_hxh[fidx]), mode="lines+markers", name="Obs phase", showlegend=False), row=2, col=1)

            fig.add_trace(go.Scatter(x=x_axis, y=np.rad2deg(pred_hxhz[fidx]), mode="lines+markers", name="Pred phase", showlegend=False), row=2, col=2)
            if obs_hxhz is not None:
                fig.add_trace(go.Scatter(x=x_axis, y=np.rad2deg(obs_hxhz[fidx]), mode="lines+markers", name="Obs phase", showlegend=False), row=2, col=2)

            for c in (1, 2):
                fig.update_xaxes(title_text=x_label, row=2, col=c)
                fig.update_yaxes(title_text="Amplitude (norm)", row=1, col=c)
                fig.update_yaxes(title_text="Phase (deg)", range=[-180.0, 180.0], row=2, col=c)

            fig.update_layout(
                title=f"Tx {tx_id}: corrected amp+phase modelled vs observed",
                height=760,
                margin=dict(t=60, b=50, l=60, r=20),
                legend=dict(orientation="h", y=-0.10),
            )
            display(fig)

        except Exception as exc:
            display(ipw.HTML(f"Data compare error: {exc}"))


try:
    load_real_btn.on_click(_old_on_load_real, remove=True)
except Exception:
    pass
load_real_btn.on_click(on_load_real)

try:
    generate_syn_btn.on_click(_old_on_generate_syn, remove=True)
except Exception:
    pass
generate_syn_btn.on_click(on_generate_syn)

push_message("Amp+phase correction patch active for results view: FD traces time-differentiated (np.gradient), FD phase (Hxx=-180°, Hxz=0°), FD 2D->3D spreading correction, shared Hxx+Hxz normalization.")